# PaperOrchestra di Google Colab (dengan API InferHub)

**Anggota kelompok (maks. 3):** ........ / ........ / ........

Sumber: [halaman proyek](https://yiwen-song.github.io/paper_orchestra/) · [kode resmi](https://github.com/google-research/paper-orchestra) · [paper arXiv 2604.05018](https://arxiv.org/abs/2604.05018)

## Konsep singkat
PaperOrchestra adalah kerangka **multi-agent** (Google Cloud AI Research) yang mengubah bahan mentah (ringkasan ide, log eksperimen, template LaTeX, panduan konferensi, gambar) menjadi draf paper LaTeX/PDF.

| Urutan | Agen | Tugas | Hasil |
|---|---|---|---|
| 1 | Outline | Menyusun rencana paper dan kebutuhan sitasi | `outline.json` |
| 2 | Plotting | Membuat diagram dan plot (*tidak dipakai di notebook ini*) | gambar |
| 3 | Literature Review | Mencari kandidat paper, memverifikasi lewat Semantic Scholar, menulis Introduction dan Related Work | `references.bib` |
| 4 | Section Writing | Menulis bagian lain, tabel, dan sisipan gambar | `raw_draft_paper.tex` |
| 5 | Content Refinement | Revisi berulang berdasarkan simulasi peer review | `final_paper.pdf` |

Menurut penulisnya, evaluasi oleh 11 peneliti menunjukkan sistem ini mengungguli *single agent* dan AI Scientist-v2, tetapi masih di bawah paper tulisan manusia. Hasilnya adalah **draf bantu**; manusia tetap bertanggung jawab memeriksa isi dan sitasi.

## Penyesuaian terhadap repo resmi
Mengisi `OPENAI_API_KEY` saja **tidak cukup**: agen Outline, Literature Review, Refinement, dan reviewer memanggil Gemini secara langsung. Notebook ini memakai adaptor `po_compat.py` yang mengalihkan panggilan tersebut ke API kompatibel OpenAI. Akibatnya:
- Google Search diganti **pengetahuan model**, lalu judul diverifikasi ke Semantic Scholar (cadangan: arXiv). Cakupan literatur bisa lebih sempit.
- PDF dikirim ke model sebagai **teks hasil ekstraksi**.
- Plotting agent **dimatikan** (butuh model gambar Gemini); gambar disediakan manual.

Jadi hasilnya **bukan replikasi** angka di paper. Sebutkan ini saat demo.

## 1. Unduh kode
Runtime CPU sudah cukup (model dijalankan di server penyedia API).

In [ ]:
%cd /content
!test -d paper-orchestra || git clone -q https://github.com/google-research/paper-orchestra.git
%cd /content/paper-orchestra
# Commit yang dipakai saat notebook ini disusun
!git checkout -q d9bae997e793a015ed38bb1605be9eb58c00284f
# Repo mencetak API key Semantic Scholar ke layar; baris itu dihapus agar tidak bocor saat demo
!sed -i '/Using S2 API Key/d' utils/scholar_utils.py
!git log -1 --oneline

## 2. Instalasi LaTeX dan library Python
Bagian LaTeX butuh beberapa menit.

In [ ]:
!apt-get update -qq
!DEBIAN_FRONTEND=noninteractive apt-get install -y -qq --no-install-recommends texlive-latex-base texlive-latex-recommended texlive-latex-extra texlive-fonts-recommended texlive-bibtex-extra texlive-science > /content/apt.log
!pip install -q -r requirements.txt
!which pdflatex bibtex

## 3. Pasang adaptor
Jalankan sel ini apa adanya; ia membuat file `po_compat.py`.

In [ ]:
%%writefile /content/paper-orchestra/po_compat.py
"""Adaptor: menjalankan PaperOrchestra dengan API yang kompatibel OpenAI.

Repo resmi memanggil Gemini secara langsung di sebagian besar agen. File ini
TIDAK mengubah kode agen; ia mengganti klien Gemini saat runtime, sehingga
semua panggilan diteruskan ke endpoint lain (NVIDIA, OpenRouter, OpenAI, dll).

Pemakaian (dari folder repo):
    python po_compat.py <argumen yang sama dengan paper_writing_cli.py>

Variabel environment:
    LLM_API_KEY        wajib
    LLM_BASE_URL       default https://integrate.api.nvidia.com/v1
    LLM_MODEL          model untuk teks
    LLM_VISION_MODEL   model untuk input gambar (kosong = gambar dilewati)
    LLM_MAX_TOKENS     default 16384
    LLM_TOKEN_PARAM    max_tokens (default) atau max_completion_tokens
    LLM_THINKING       on / off / kosong (khusus model Nemotron di NVIDIA)
    LLM_RPM            batas request per menit, default 30
    PO_MAX_SEARCH_TASKS  batasi jumlah pencarian literatur (0 = semua)
"""

import base64
import os
import re
import sys
import threading
import time
import xml.etree.ElementTree as ET
from types import SimpleNamespace

sys.path.insert(0, os.getcwd())

import cv2
import numpy as np
import pymupdf
import requests
from openai import APIConnectionError, APIStatusError, OpenAI
from thefuzz import fuzz

BASE_URL = os.getenv("LLM_BASE_URL") or "https://integrate.api.nvidia.com/v1"
MODEL = os.getenv("LLM_MODEL") or "nvidia/nemotron-3.5-super-vl-120b-a12b"
VISION_MODEL = os.getenv("LLM_VISION_MODEL", "")
MAX_TOKENS = int(os.getenv("LLM_MAX_TOKENS", "16384"))
TOKEN_PARAM = os.getenv("LLM_TOKEN_PARAM", "max_tokens")
THINKING = os.getenv("LLM_THINKING", "").lower()
RPM = int(os.getenv("LLM_RPM", "30"))
MAX_SEARCH_TASKS = int(os.getenv("PO_MAX_SEARCH_TASKS", "0"))

SEARCH_NOTE = (
    "NOTE: No web search tool is available. Answer from your own knowledge and "
    "list only papers whose exact title you are confident really exists. "
    "Every title will be verified against an academic database."
)

_rate_lock = threading.Lock()
_next_slot = 0.0
_slots = threading.Semaphore(3)  # maksimal 3 request bersamaan
stats = {"calls": 0, "prompt_tokens": 0, "completion_tokens": 0}


def _wait_for_rate_limit():
    global _next_slot
    with _rate_lock:
        now = time.time()
        start = max(now, _next_slot)
        _next_slot = start + 60.0 / RPM
    time.sleep(max(0.0, start - now))


def _image_part(data: bytes) -> dict:
    """Perkecil gambar lalu kirim sebagai data URL JPEG."""
    img = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError("Gambar tidak bisa dibaca")
    scale = 1400 / max(img.shape[:2])
    if scale < 1:
        img = cv2.resize(img, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    ok, buf = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 80])
    b64 = base64.b64encode(buf.tobytes()).decode()
    return {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}}


def _convert(contents):
    """Ubah 'contents' gaya Gemini menjadi (daftar teks, daftar gambar)."""
    if not isinstance(contents, (list, tuple)):
        contents = [contents]
    texts, images = [], []
    for part in contents:
        if isinstance(part, str):
            texts.append(part)
        elif getattr(part, "text", None) is not None:
            texts.append(part.text)
        elif getattr(part, "inline_data", None) is not None:
            blob = part.inline_data
            if blob.mime_type == "application/pdf":
                with pymupdf.open(stream=blob.data, filetype="pdf") as doc:
                    text = "\n".join(page.get_text() for page in doc)
                texts.append("--- TEXT EXTRACTED FROM THE CURRENT PDF ---\n" + text)
            else:
                images.append(_image_part(blob.data))
        else:
            raise TypeError(f"Jenis konten tidak dikenal: {type(part)}")
    return texts, images


def _clean(text: str) -> str:
    """Buang blok penalaran <think>...</think> bila model menyertakannya."""
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    return text.split("</think>")[-1].strip()


class _Models:
    def __init__(self):
        self.client = OpenAI(
            api_key=os.environ["LLM_API_KEY"],
            base_url=BASE_URL,
            timeout=900,
            max_retries=0,
        )

    def generate_content(self, model, contents, config=None):
        texts, images = _convert(contents)
        system = getattr(config, "system_instruction", None)
        temperature = getattr(config, "temperature", None)
        if getattr(config, "tools", None):
            texts.append(SEARCH_NOTE)
        prompt = "\n\n".join(texts)

        use_model = MODEL
        if images:
            if VISION_MODEL:
                use_model = VISION_MODEL
            elif "formatting issues" in prompt:
                # Tanpa model vision, pemeriksaan tata letak dilewati.
                return SimpleNamespace(
                    text='{"figure_and_tables": {}, "other_issues": []}'
                )
            else:
                images = []

        content = prompt
        if images:
            content = [{"type": "text", "text": prompt}] + images[:8]
        messages = [{"role": "user", "content": content}]
        if system:
            messages.insert(0, {"role": "system", "content": str(system)})

        kwargs = {TOKEN_PARAM: MAX_TOKENS}
        if temperature is not None and TOKEN_PARAM == "max_tokens":
            kwargs["temperature"] = temperature
        if THINKING in ("on", "off"):
            kwargs["extra_body"] = {
                "chat_template_kwargs": {"enable_thinking": THINKING == "on"}
            }

        for attempt in range(6):
            try:
                return self._stream(use_model, messages, kwargs)
            except (APIStatusError, APIConnectionError) as e:
                code = getattr(e, "status_code", None)
                if code is not None and code != 429 and code < 500:
                    raise
                if attempt == 5:
                    raise
                wait = min(15 * 2**attempt, 120)
                print(f"   [adaptor] server sibuk ({code}), tunggu {wait} detik")
                time.sleep(wait)

    def _stream(self, model, messages, kwargs):
        with _slots:
            _wait_for_rate_limit()
            started = time.time()
            chunks, finish, usage = [], None, None
            stream = self.client.chat.completions.create(
                model=model, messages=messages, stream=True, **kwargs
            )
            for chunk in stream:
                usage = getattr(chunk, "usage", None) or usage
                if not chunk.choices:
                    continue
                choice = chunk.choices[0]
                if choice.delta and choice.delta.content:
                    chunks.append(choice.delta.content)
                finish = choice.finish_reason or finish

        text = _clean("".join(chunks))
        stats["calls"] += 1
        if usage:
            stats["prompt_tokens"] += usage.prompt_tokens or 0
            stats["completion_tokens"] += usage.completion_tokens or 0
        print(
            f"   [adaptor] call #{stats['calls']} {model} "
            f"{time.time() - started:.0f}s, {len(text)} karakter"
        )
        if finish == "length":
            raise RuntimeError(
                "Output terpotong. Naikkan LLM_MAX_TOKENS atau set LLM_THINKING=off."
            )
        if not text:
            raise RuntimeError("Model mengembalikan jawaban kosong.")
        return SimpleNamespace(text=text)


# --- Pencarian paper: Semantic Scholar, lalu arXiv sebagai cadangan ---

_arxiv_lock = threading.Lock()


def _arxiv_search(title):
    words = re.sub(r"[^A-Za-z0-9 ]", " ", title)
    with _arxiv_lock:
        time.sleep(3)  # etika pemakaian API arXiv
        rsp = requests.get(
            "https://export.arxiv.org/api/query",
            params={"search_query": f'ti:"{" ".join(words.split())}"', "max_results": 3},
            timeout=20,
        )
    rsp.raise_for_status()
    ns = {"a": "http://www.w3.org/2005/Atom"}
    best, best_ratio = None, 70
    for entry in ET.fromstring(rsp.content).findall("a:entry", ns):
        found = " ".join(entry.findtext("a:title", "", ns).split())
        ratio = fuzz.ratio(title.lower(), found.lower())
        if ratio > best_ratio:
            date = entry.findtext("a:published", "", ns)[:10]
            best_ratio = ratio
            best = {
                "title": found,
                "authors": [
                    {"name": a.findtext("a:name", "", ns)}
                    for a in entry.findall("a:author", ns)
                ],
                "venue": "arXiv",
                "year": int(date[:4]),
                "publicationDate": date,
                "abstract": " ".join(entry.findtext("a:summary", "", ns).split()),
                "citationCount": 0,
            }
    return best


def _make_search(original):
    def search(title, year_hint=None, cutoff_date=None):
        result = original(title, year_hint, cutoff_date)
        if result and result.get("abstract"):
            return result
        try:
            return _arxiv_search(title)
        except Exception as e:
            print(f"   [adaptor] arXiv gagal untuk '{title[:40]}': {e}")
            return None

    return search


def install():
    from methods.agents import literature_review_agent as lit
    from utils import gemini_utils

    gemini_utils._genai_client = SimpleNamespace(models=_Models())
    lit.s2_title_search = _make_search(lit.s2_title_search)

    if MAX_SEARCH_TASKS > 0:
        collect = lit.HybridLiteratureAgent._collect_search_tasks

        def limited(self, outline):
            tasks = collect(self, outline)
            if len(tasks) <= MAX_SEARCH_TASKS:
                return tasks
            step = len(tasks) / MAX_SEARCH_TASKS
            return [tasks[int(i * step)] for i in range(MAX_SEARCH_TASKS)]

        lit.HybridLiteratureAgent._collect_search_tasks = limited


def main():
    if not os.getenv("LLM_API_KEY"):
        sys.exit("LLM_API_KEY belum diisi.")
    if re.search(r"--use_plotting[= ](true|1|yes)", " ".join(sys.argv), re.I):
        sys.exit("Plotting agent butuh model gambar Gemini; pakai --use_plotting false.")

    install()
    import paper_writing_cli

    # Nama model hanya label; model sebenarnya diambil dari LLM_MODEL.
    sys.argv += ["--writer_model_name", "compat", "--reflection_model_name", "compat"]
    start = time.time()
    paper_writing_cli.main()
    print(f"\n[adaptor] selesai dalam {(time.time() - start) / 60:.1f} menit: {stats}")


if __name__ == "__main__":
    main()

## 4. API key dan model
1. Di Colab, buka ikon kunci (**Secrets**), tambah `INFERHUB_API_KEY` (diawali `sk-airo-`), aktifkan *Notebook access*.
2. Opsional: tambah `SEMANTIC_SCHOLAR_API_KEY` agar pencarian sitasi lebih stabil.

Model teks dan gambar sudah diatur di bawah. InferHub menagih per token, pantau saldo. **Jangan menulis key langsung di sel.**

In [ ]:
import os
from getpass import getpass

def secret(name, required=True):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    return getpass(f"Masukkan {name} (Enter untuk lewati): " if not required else f"Masukkan {name}: ")

os.environ.update({
    "LLM_API_KEY": secret("INFERHUB_API_KEY").strip(),
    "LLM_BASE_URL": "https://api.inferhub.dev/v1",
    "LLM_MODEL": "alicn/deepseek-v4-pro-0813",         # teks
    "LLM_VISION_MODEL": "ag/gemini-3.8-flash-high",    # gambar
    "LLM_THINKING": "",
    "LLM_TOKEN_PARAM": "max_tokens",
    "LLM_MAX_TOKENS": "32768",
    "LLM_RPM": "30",
    "PO_MAX_SEARCH_TASKS": "8",                        # 0 = semua tugas pencarian
})
s2 = secret("SEMANTIC_SCHOLAR_API_KEY", required=False).strip()
if s2:
    os.environ["SEMANTIC_SCHOLAR_API_KEY"] = s2
print("Teks:", os.environ["LLM_MODEL"], "| Gambar:", os.environ["LLM_VISION_MODEL"])

## 5. Tes koneksi
Keduanya harus terjawab (gambar: lingkaran merah). Jika model tidak tersedia, ganti namanya di sel 4 lalu jalankan ulang sel 4 dan 5.

In [ ]:
import base64, cv2, numpy as np
from openai import OpenAI

client = OpenAI(api_key=os.environ["LLM_API_KEY"], base_url=os.environ["LLM_BASE_URL"], timeout=300)

def ask(model, content):
    r = client.chat.completions.create(
        model=model, messages=[{"role": "user", "content": content}], max_tokens=2000)
    return r.choices[0].message.content

print("Teks  :", ask(os.environ["LLM_MODEL"], "Balas dengan satu kata: siap"))
img = np.full((200, 200, 3), 255, np.uint8)
cv2.circle(img, (100, 100), 60, (0, 0, 255), -1)
url = "data:image/jpeg;base64," + base64.b64encode(cv2.imencode(".jpg", img)[1]).decode()
print("Gambar:", ask(os.environ["LLM_VISION_MODEL"], [
    {"type": "text", "text": "Bentuk dan warna apa ini? Jawab singkat."},
    {"type": "image_url", "image_url": {"url": url}}]))

## 6. Siapkan bahan mentah
Sumber: Putri, Nurramdhani & Hartati (2026), *A Retrieval-Augmented Generation Framework for a Study Program Accreditation Question-Answering System*, ETASR 16(2):33375-33381, [doi:10.48084/etasr.16326](https://doi.org/10.48084/etasr.16326), CC BY 4.0.

Sel ini membuat ringkasan ide, log eksperimen, dan dua gambar yang digambar ulang. Keluaran pipeline adalah tulisan ulang riset orang lain: **hanya untuk latihan**.

In [ ]:
# Bahan mentah demo, diringkas dari: Putri, Nurramdhani & Hartati (2026),
# ETASR 16(2):33375-33381, doi:10.48084/etasr.16326, lisensi CC BY 4.0.
# Jalankan di sel Colab (folder /content/paper-orchestra).
import json, pathlib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

demo = pathlib.Path("/content/paper-orchestra/demo_rag")
(demo / "figures").mkdir(parents=True, exist_ok=True)

(demo / "idea.md").write_text("""## Working Topic
A Retrieval-Augmented Generation (RAG) framework for an Indonesian-language question-answering system on study program accreditation.

## Problem Statement
Accreditation regulations issued by LAM INFOKOM (Indonesia's accreditation agency for informatics and computing) are long and technical. Applicants struggle to interpret them and staff receive repetitive inquiries. In a regulatory setting, answers must be strictly grounded in authoritative text. Prior RAG work mostly targets English or open-domain data.

## Core Hypothesis
Semantic retrieval over article-level regulatory units, combined with a generator restricted to the retrieved text, gives accurate and traceable answers without fine-tuning.

## Methodology
1. Corpus: 42 LAM INFOKOM regulatory documents (2022-2025), manually segmented by two annotators into 203 articles/decrees. Headers, footers and duplicated formatting are removed. Each unit stores document number, title, article number, decree number, text, topic annotation and source URL.
2. Retrieval: cosine similarity with four sentence embedding models (MiniLM, E5, IndoSBERT, BGE-M3). Variants: with or without preprocessing (cleaned text, titles, topic annotations) and with or without windowing (up to 110 words, 25-word overlap, mean pooling).
3. Baseline: fuzzy string matching with RapidFuzz (token_set_ratio, token_sort_ratio, partial_ratio), taking the highest score.
4. Generation: GPT-OSS 20B with in-context learning, no fine-tuning. The prompt holds the question, the retrieved excerpts and an instruction to answer only from them, in Indonesian.
5. Prototype: Gradio interface showing the answer together with the retrieved excerpts.

## Expected Contribution
An end-to-end, traceable QA pipeline for a low-resource regulatory domain, with a comparison of retrievers and generators.

## Limitations
No domain fine-tuning; tailored to Indonesian and LAM INFOKOM; retrieval errors propagate to the answers.

## Future Work
Larger dataset, fine-tuned open models, conversational memory, other accreditation bodies.
""")

(demo / "experimental_log.md").write_text("""## 1. Experimental Setup
* Reference corpus: 42 documents, 203 articles/decrees.
* Evaluation set: 150 questions from applicant FAQs that refer to specific clauses or topics, plus manual paraphrases.
* Retrieval metrics: Precision@3, Recall@3, MRR.
* Generation metrics: BERTScore precision, recall, F1 against reference answers.
* Human evaluation: 50 randomly selected samples; factual correctness and naturalness (5-point Likert).

## 2. Raw Numeric Data

### Corpus statistics
| Year | Documents | Articles/decrees |
|---|---|---|
| 2022 | 9 | 69 |
| 2023 | 16 | 99 |
| 2024 | 7 | 24 |
| 2025 | 10 | 11 |
| Total | 42 | 203 |

### Retrieval performance
| Model | Preprocessing | Windowing | Prec@3 | Recall@3 | MRR |
|---|---|---|---|---|---|
| Fuzzy | no | no | 0.193 | 0.580 | 0.757 |
| IndoSBERT | no | no | 0.227 | 0.680 | 0.549 |
| MiniLM | no | no | 0.211 | 0.633 | 0.591 |
| E5 | no | no | 0.264 | 0.793 | 0.694 |
| BGE-M3 | no | no | 0.289 | 0.867 | 0.805 |
| E5 | yes | yes | 0.287 | 0.860 | 0.759 |
| BGE-M3 | yes | yes | 0.293 | 0.880 | 0.804 |
| E5 | yes | no | 0.304 | 0.913 | 0.828 |
| BGE-M3 | yes | no | 0.309 | 0.927 | 0.881 |

### Generator comparison (retriever fixed to BGE-M3)
| Model | Parameters | Precision | Recall | F1 |
|---|---|---|---|---|
| LLaMA 3.1 | 8B | 0.717 | 0.797 | 0.753 |
| Gemma 3 | 27B | 0.718 | 0.803 | 0.756 |
| Gemma-SEA-LION-v4 | 27B | 0.711 | 0.804 | 0.752 |
| GPT-OSS | 20B | 0.729 | 0.805 | 0.763 |

### Retriever effect on answers (generator fixed to GPT-OSS)
| Retriever | Precision | Recall | F1 |
|---|---|---|---|
| Fuzzy | 0.718 | 0.786 | 0.748 |
| IndoSBERT | 0.680 | 0.684 | 0.681 |
| MiniLM | 0.717 | 0.779 | 0.745 |
| BGE-M3 | 0.729 | 0.805 | 0.763 |

### Manual evaluation
* Factually correct and aligned with the regulation text: 83.67% (reported for 50 samples).
* Average naturalness: 4.3 / 5.

## 3. Qualitative Observations
* BGE-M3 with preprocessing and no windowing is the best retriever; windowing did not help.
* On an appeal-fee question, both BGE-M3 and fuzzy matching retrieved the correct clause.
* On a re-accreditation timing question, BGE-M3 was correct while fuzzy matching returned an unrelated clause.
* GPT-OSS 20B had the highest BERTScore F1 although smaller than the Gemma models; differences are small.
* BERTScore measures similarity, not factual correctness.
""")

# Gambar 1: alur sistem (digambar ulang dari uraian metode)
fig, ax = plt.subplots(figsize=(11, 3.2))
ax.set_xlim(0, 11)
ax.set_ylim(0, 3.2)
ax.axis("off")

def box(x, y, text):
    ax.add_patch(FancyBboxPatch((x, y), 1.9, 0.8, boxstyle="round,pad=0.05",
                                fc="#e8f0fe", ec="#1a4a8a"))
    ax.text(x + 0.95, y + 0.4, text, ha="center", va="center", fontsize=9)

def arrow(a, b):
    ax.annotate("", xy=b, xytext=a, arrowprops=dict(arrowstyle="->"))

box(0.1, 2.1, "42 regulation\ndocuments")
box(2.5, 2.1, "Segmentation\n(203 units)")
box(4.9, 2.1, "Embedding\nindex")
box(0.1, 0.3, "User question")
box(4.9, 0.3, "Cosine similarity\nretrieval (top-k)")
box(7.35, 0.3, "Generator\n(GPT-OSS 20B)")
box(9.0, 2.1, "Answer +\nsource excerpts")
arrow((2.05, 2.5), (2.45, 2.5))
arrow((4.45, 2.5), (4.85, 2.5))
arrow((5.85, 2.05), (5.85, 1.15))
arrow((2.05, 0.7), (4.85, 0.7))
arrow((6.85, 0.7), (7.3, 0.7))
arrow((8.7, 1.15), (9.7, 2.05))
fig.savefig(demo / "figures" / "workflow.png", dpi=200, bbox_inches="tight")
plt.close(fig)

# Gambar 2: MRR per konfigurasi
names = ["Fuzzy", "IndoSBERT", "MiniLM", "E5", "BGE-M3", "E5\n+prep+win",
         "BGE-M3\n+prep+win", "E5\n+prep", "BGE-M3\n+prep"]
mrr = [0.757, 0.549, 0.591, 0.694, 0.805, 0.759, 0.804, 0.828, 0.881]
fig, ax = plt.subplots(figsize=(11, 3.5))
bars = ax.bar(names, mrr, color="#4a78b5")
ax.bar_label(bars, fmt="%.3f", fontsize=8)
ax.set_ylabel("MRR")
ax.set_ylim(0, 1)
ax.tick_params(axis="x", labelsize=8)
fig.savefig(demo / "figures" / "mrr.png", dpi=200, bbox_inches="tight")
plt.close(fig)

json.dump([
    {"name": "workflow.png", "caption": "Workflow of the retrieval-augmented question-answering system."},
    {"name": "mrr.png", "caption": "Mean Reciprocal Rank of each retrieval configuration."},
], open(demo / "figures" / "info.json", "w"), indent=2)
print("Bahan siap di", demo)

## 7. Jalankan pipeline
Puluhan panggilan API; **jalankan jauh sebelum demo** karena bisa lama (tergantung antrean server). Biarkan tab Colab terbuka.

Bila satu tahap gagal, repo mengulang seluruh pipeline sampai 3 kali. Jika error yang sama terus muncul, hentikan sel dan lihat tabel masalah di bawah.

In [ ]:
%cd /content/paper-orchestra
!python -u po_compat.py \
  --raw_materials_dir demo_rag \
  --latex_template_dir templates/cvpr2025 \
  --output_dir output_rag \
  --idea_filename idea.md \
  --experimental_log_filename experimental_log.md \
  --research_cutoff 2025-11 \
  --use_plotting false 2>&1 | tee run_rag.log

| Gejala | Tindakan |
|---|---|
| `401` / `403` | Key salah atau akun belum terverifikasi |
| `404` model | Ganti nama model di sel 4 |
| `402` / saldo kurang | Isi saldo atau pilih model gratis |
| `400` soal `temperature` / `max_tokens` | Pilih model lain |
| `429` berulang | Turunkan `LLM_RPM` ke 15, tunggu beberapa menit |
| `Output terpotong` | Naikkan `LLM_MAX_TOKENS` (mis. 32768) |
| `Parsed response is None` | Model tidak mengembalikan JSON valid; ulangi atau ganti model |
| `PDF Generation Failed` | Baca log LaTeX di atasnya; jalankan ulang |
| Referensi 0 | Pasang key Semantic Scholar, ulangi dari sel 4 |

## 8. Periksa dan tampilkan hasil

In [ ]:
import pymupdf, os, json
from IPython.display import display, Image

assert os.path.exists("output_rag/final_paper.pdf"), "PDF belum jadi, periksa run_rag.log"
refs = json.load(open("output_rag/literature_agent_output/citation_map.json"))
print("Jumlah referensi terverifikasi:", len(refs))
if not refs:
    print("PERINGATAN: tidak ada referensi; run belum bisa disebut tuntas.")
for r in list(refs.values())[:10]:
    print("-", r["year"], r["title"])

for page in pymupdf.open("output_rag/final_paper.pdf"):
    display(Image(data=page.get_pixmap(dpi=80).tobytes("png")))

In [ ]:
# Jejak tiap agen (bagus untuk ditunjukkan saat demo)
!head -c 1500 output_rag/outline.json; echo
!ls output_rag/literature_agent_output output_rag/content_refinement_workdir output_rag/content_refinement_workdir/peer_reviews
!cat output_rag/content_refinement_workdir/content_refinement_worklog.json | head -60

In [ ]:
!zip -qr hasil_rag.zip output_rag run_rag.log demo_rag
from google.colab import files
files.download("hasil_rag.zip")

## Panduan demo (sekitar 10 menit)
1. **Masalah dan konsep** (2 menit): bahan mentah menjadi draf paper; tabel lima agen.
2. **Instalasi** (2 menit): sel 1-5, jelaskan mengapa perlu adaptor.
3. **Input** (1 menit): paper sumber, `idea.md`, `experimental_log.md`, gambar, template.
4. **Hasil** (3 menit): tampilkan PDF, `outline.json`, referensi, skor peer review. Pakai hasil yang sudah dijalankan sebelumnya.
5. **Evaluasi kritis** (2 menit): bandingkan angka di tabel PDF dengan log eksperimen, cek 3 sitasi apakah relevan, sebutkan keterbatasan dan etika.

**Temuan untuk dikritisi** (cocokkan dulu dengan PDF asli)
- Akurasi manual 83,67% dari 50 sampel bukan bilangan bulat; cara hitungnya tidak jelas.
- Baris Fuzzy: MRR 0,757 dengan Recall@3 0,580. Jika satu jawaban relevan per pertanyaan, MRR maksimal sekitar 0,685.

**Pertanyaan yang mungkin muncul**
- *Bedanya dengan chatbot biasa?* Tugas dipecah ke beberapa agen, sitasi diverifikasi, dan ada siklus review-revisi.
- *Apakah sitasi bisa halusinasi?* Judul dicocokkan ke basis data, tetapi kesesuaian isi sitasi tetap harus diperiksa manusia.
- *Mengapa tidak pakai Gemini gratis?* Kuotanya tidak cukup untuk run penuh.
- *Apa itu InferHub?* Perantara (proxy) berbayar per token ke berbagai model; data dikirim lewat pihak ketiga.